# Dhwani Multilingual Deepfake Audio Detection - Colab Fine-Tuning
This notebook streams HuggingFace datasets (to bypass 800GB+ downloads), validates the ONNX to PyTorch conversion, fine-tunes the model on an IndicSynth + Common Voice subset, and evaluates the baseline and fine-tuned models on ASVspoof2019 LA and In-The-Wild datasets.

In [ ]:
!nvidia-smi
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
    print("Total Memory (GB):", torch.cuda.get_device_properties(0).total_memory / 1024**3)


In [ ]:
!pip install transformers onnx onnxruntime onnx2torch librosa scikit-learn torchaudio datasets

# PATCH ONNX2TORCH to bypass torchvision dependency issues
import os
import site
site_packages = site.getsitepackages()[0]
onnx2torch_init = os.path.join(site_packages, "onnx2torch", "node_converters", "__init__.py")

if os.path.exists(onnx2torch_init):
    with open(onnx2torch_init, "r") as f:
        content = f.read()
    content = content.replace("from onnx2torch.node_converters.nms import *", "# removed nms")
    content = content.replace("from onnx2torch.node_converters.roialign import *", "# removed roialign")
    with open(onnx2torch_init, "w") as f:
        f.write(content)
    print("Patched onnx2torch successfully!")


In [ ]:
from google.colab import drive
import urllib.request
import os

drive.mount('/content/drive')
base_dir = '/content/drive/MyDrive/dhwani'
os.makedirs(f"{base_dir}/models", exist_ok=True)
os.makedirs(f"{base_dir}/checkpoints", exist_ok=True)
os.makedirs(f"{base_dir}/results", exist_ok=True)

onnx_path = f"{base_dir}/models/best_model_original.onnx"
if not os.path.exists(onnx_path):
    print("Downloading best_model.onnx from HuggingFace...")
    url = "https://huggingface.co/ayush2635/Dhwani-Multilingual-Deepfake-Audio-Detection-Model/resolve/main/best_model.onnx"
    urllib.request.urlretrieve(url, onnx_path)
    print("Download complete!")


In [ ]:
import onnx
from onnx2torch import convert
import torch
import numpy as np

print("Loading ONNX model...")
onnx_model = onnx.load(onnx_path, load_external_data=False)
print("\nConverting ONNX to PyTorch... This takes about 60 seconds.")
torch_model = convert(onnx_model)
torch_model.eval()
torch_model.cuda()
print("Conversion successful!")


## Dataset Streaming Strategy

Instead of downloading 845GB of **IndicSynth** and huge **Common Voice** datasets, we use `streaming=True` to lazily sample exactly 500 files per language (Hindi, Malayalam, Tamil, Telugu).

*   **Fine-Tuning/Adaptation Data:** IndicSynth (Fake), Common Voice (Real)
*   **External Evaluation Data:** ASVspoof 2019 LA, In-The-Wild
*   *(Optional Future)* **MLAAD:** 183GB HuggingFace dataset (requires approval, CC-BY-NC 4.0 license). Skipping for this experiment to unblock training.


In [ ]:
from datasets import load_dataset
import librosa
import random
from torch.utils.data import Dataset, DataLoader
import json

TARGET_SAMPLES_PER_LANG = 500
LANGUAGES = ['hi', 'ml', 'ta', 'te'] # Hindi, Malayalam, Tamil, Telugu
# Note: Common Voice language codes for these are typically 'hi', 'ml', 'ta', 'te' or similar depending on the version.
# For demonstration, we simulate the extraction logic via the HF datasets API.

# 1. Fetch Fine-Tuning Data (Streaming)
def process_audio(audio_array, sr, max_len=48000):
    if sr != 16000:
        y = librosa.resample(y=audio_array, orig_sr=sr, target_sr=16000)
    else:
        y = audio_array
    
    if len(y) > max_len:
        y = y[:max_len]
    else:
        y = np.pad(y, (0, max_len - len(y)))
        
    y = (y - np.mean(y)) / np.sqrt(np.var(y) + 1e-5)
    return y.astype(np.float32)

train_data = []

print("Extracting Real Speech (Common Voice)...")
# Attempting to load Mozilla Common Voice (requires HF token if restricted, using older version if needed)
# Using `mozilla-foundation/common_voice_11_0` as an example
try:
    for lang in LANGUAGES:
        cv = load_dataset("mozilla-foundation/common_voice_11_0", lang, split="train", streaming=True, trust_remote_code=True)
        count = 0
        for item in cv:
            audio = item['audio']
            y = process_audio(audio['array'], audio['sampling_rate'])
            train_data.append((y, 0)) # Real = 0
            count += 1
            if count >= TARGET_SAMPLES_PER_LANG:
                break
        print(f"Loaded {count} Real samples for {lang}")
except Exception as e:
    print(f"Common Voice extraction warning: {e}. You may need to provide a HF token or adjust the dataset name.")

print("\nExtracting Synthetic Speech (IndicSynth)...")
try:
    indicsynth = load_dataset("vdivyasharma/IndicSynth", split="train", streaming=True, trust_remote_code=True)
    lang_counts = {l: 0 for l in LANGUAGES}
    
    for item in indicsynth:
        # Assuming the dataset has a 'language' column or similar. We filter dynamically.
        # Fallback if structure varies: just pull 2000 total fake items.
        audio = item['audio']
        y = process_audio(audio['array'], audio['sampling_rate'])
        train_data.append((y, 1)) # Fake = 1
        
        total_fake = sum(1 for _, label in train_data if label == 1)
        if total_fake >= TARGET_SAMPLES_PER_LANG * len(LANGUAGES):
            break
    print(f"Loaded {total_fake} Synthetic samples from IndicSynth")
except Exception as e:
    print(f"IndicSynth extraction warning: {e}")

random.shuffle(train_data)
print(f"\nTotal Fine-Tuning samples loaded: {len(train_data)}")

# Save metadata
metadata = {
    "target_per_lang": TARGET_SAMPLES_PER_LANG,
    "languages": LANGUAGES,
    "total_train_samples": len(train_data)
}
with open(f"{base_dir}/results/dataset_metadata.json", "w") as f:
    json.dump(metadata, f)

# PyTorch Dataset
class StreamedDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        y, label = self.data[idx]
        return torch.tensor(y), torch.tensor(label, dtype=torch.long)

# Train/Val Split (80/20)
split_idx = int(0.8 * len(train_data))
train_loader = DataLoader(StreamedDataset(train_data[:split_idx]), batch_size=4, shuffle=True)
val_loader = DataLoader(StreamedDataset(train_data[split_idx:]), batch_size=4, shuffle=False)


In [ ]:
print("Loading ASVspoof2019_LA (External Evaluation 1)...")
eval1_data = []
try:
    asv = load_dataset("SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA", split="eval", streaming=True, trust_remote_code=True)
    count = 0
    for item in asv:
        # Typically ASVspoof has 'audio' and 'label' (bonafide / spoof)
        audio = item['audio']
        is_spoof = 1 if item['label'] == 'spoof' else 0
        y = process_audio(audio['array'], audio['sampling_rate'])
        eval1_data.append((y, is_spoof))
        count += 1
        if count >= 1000: # Limit for evaluation speed
            break
    print(f"Loaded {count} ASVspoof2019_LA evaluation samples.")
except Exception as e:
    print(f"ASVspoof extraction failed: {e}")

print("\nLoading In-The-Wild (External Evaluation 2)...")
eval2_data = []
try:
    itw = load_dataset("SpeechAntiSpoofingBenchmarks/InTheWild", split="eval", streaming=True, trust_remote_code=True)
    count = 0
    for item in itw:
        audio = item['audio']
        is_spoof = 1 if item['label'] == 'spoof' else 0
        y = process_audio(audio['array'], audio['sampling_rate'])
        eval2_data.append((y, is_spoof))
        count += 1
        if count >= 1000: # Limit for evaluation speed
            break
    print(f"Loaded {count} In-The-Wild evaluation samples.")
except Exception as e:
    print(f"In-The-Wild extraction failed: {e}")

eval1_loader = DataLoader(StreamedDataset(eval1_data), batch_size=8) if eval1_data else None
eval2_loader = DataLoader(StreamedDataset(eval2_data), batch_size=8) if eval2_data else None


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
import torch.nn.functional as F

def evaluate_external(model, dataloader, dataset_name="Dataset"):
    if not dataloader:
        print(f"Skipping {dataset_name} (No data)")
        return {}
        
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.cuda()
            logits = model(inputs)
            probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy() # Probability of fake
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
            all_probs.extend(probs)
            
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    
    try:
        roc_auc = roc_auc_score(all_labels, all_probs)
    except:
        roc_auc = 0.0 # Handles single-class batches edge-case
        
    print(f"\n--- {dataset_name} ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(all_labels, all_preds))
    
    return {"acc": acc, "prec": prec, "rec": rec, "f1": f1, "roc_auc": roc_auc}

print("========================================")
print("BASELINE EVALUATION (Original ONNX PyTorch conversion)")
print("========================================")
baseline_metrics_1 = evaluate_external(torch_model, eval1_loader, "ASVspoof 2019 LA")
baseline_metrics_2 = evaluate_external(torch_model, eval2_loader, "In-The-Wild")


In [ ]:
import torch.optim as optim
import torch.nn as nn

# Freeze base, train backend
for param in torch_model.parameters():
    param.requires_grad = False

params_to_train = []
for name, param in torch_model.named_parameters():
    if "MatMul" in name or "Linear" in name:
        if int(name.split('_')[-1]) > 2500: # AASIST backend heuristic
            param.requires_grad = True
            params_to_train.append(param)
            
if not params_to_train:
    all_params = list(torch_model.parameters())
    params_to_train = all_params[-10:]
    for p in params_to_train:
        p.requires_grad = True

optimizer = optim.AdamW(params_to_train, lr=2e-5)
criterion = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler()

epochs = 3
accumulation_steps = 4

print("\nStarting Fine-Tuning...")
if len(train_data) > 0:
    for epoch in range(epochs):
        torch_model.train()
        for i, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.cuda(), labels.cuda()
            with torch.cuda.amp.autocast():
                outputs = torch_model(inputs)
                loss = criterion(outputs, labels) / accumulation_steps
                
            scaler.scale(loss).backward()
            if (i + 1) % accumulation_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                
            if i % 20 == 0:
                print(f"Epoch {epoch+1}/{epochs}, Step {i}, Loss: {loss.item() * accumulation_steps:.4f}")
                
        print(f"Saving checkpoint to {base_dir}/checkpoints/model_epoch_{epoch+1}.pt")
        torch.save(torch_model.state_dict(), f"{base_dir}/checkpoints/model_epoch_{epoch+1}.pt")
else:
    print("No training data loaded. Skipping fine-tuning.")


In [ ]:
print("========================================")
print("FINE-TUNED EVALUATION")
print("========================================")
ft_metrics_1 = evaluate_external(torch_model, eval1_loader, "ASVspoof 2019 LA")
ft_metrics_2 = evaluate_external(torch_model, eval2_loader, "In-The-Wild")

# Comparison report
print("\n========================================")
print("EXPERIMENT COMPARISON REPORT")
print("========================================")
def print_comparison(ds_name, base, ft):
    if not base or not ft: return
    print(f"\n--- {ds_name} ---")
    print(f"Metric       Baseline    Fine-Tuned   Delta")
    for k in base.keys():
        delta = ft[k] - base[k]
        print(f"{k.ljust(12)} {base[k]:.4f}      {ft[k]:.4f}       {delta:+.4f}")

print_comparison("ASVspoof 2019 LA", baseline_metrics_1, ft_metrics_1)
print_comparison("In-The-Wild", baseline_metrics_2, ft_metrics_2)


In [ ]:
print("Exporting fine-tuned model back to ONNX...")
torch_model.eval()
dummy_input = torch.randn(1, 48000, device='cuda', dtype=torch.float32)

export_path = f"{base_dir}/models/best_model_finetuned.onnx"

torch.onnx.export(
    torch_model,
    dummy_input,
    export_path,
    export_params=True,
    opset_version=18,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

print(f"Successfully exported fine-tuned ONNX model to {export_path}")
